## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [35]:
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

pd.set_option('display.max_columns', None)

from src.get_tables import tickers
from src.model_functions import build_lstm_sequences, split_temporal, generate_lstm_model

## Configurações

In [ ]:
tabela_hiperparams = pd.read_excel("dados/refined/tabela_melhores_hiperparametros.xlsx")

features = [
    'close', 'high', 'low', 'open', 'volume',
    'close_dolar', 'close_ibovespa', 'close_sp_500', 'selic', 'ipca',
    'ma20', 'ma50', 'bb_upper', 'bb_lower', 'rsi_wilder', 'macd',
    'macd_signal', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos'
]

In [36]:
def smape(y_true, y_pred):
    return 100 * np.mean(
        2 * np.abs(y_pred - y_true) /
        (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    )

## Base de dados

In [39]:
resultados = []

for ticker in tqdm(tickers):
    print(f"Processando para {ticker}")

    df = pd.read_parquet(f"dados/refined/tb_analitica_{ticker}.parquet")
    df = df[df['close'].notna()]
    df = df.sort_values("date").reset_index(drop=True)

    hiperparams = tabela_hiperparams[tabela_hiperparams['ticker'] == ticker].drop(columns = ['ticker', 'val_loss']).to_dict(orient = 'records')[0]

    for k, v in hiperparams.items():
        globals()[k] = v

    media_close = df.close.mean()
    mediana_close = df.close.median()
    desvio_close = df.close.std()
    cv = desvio_close / media_close * 100

    data = df[features].copy()

    # Divisão temporal ANTES do scaler (evita data leakage)
    n = len(data)
    train_end = int(n * 0.70)
    valid_end = int(n * 0.85)

    train_df = data.iloc[:train_end]
    valid_df = data.iloc[train_end:valid_end]
    test_df = data.iloc[valid_end:]

    scaler = MinMaxScaler()
    scaler.fit(train_df)

    scaled = np.vstack([
        scaler.transform(train_df),
        scaler.transform(valid_df),
        scaler.transform(test_df)
    ])

    X, y = build_lstm_sequences(window_size, scaled)
    X_train, y_train, X_valid, y_valid, X_test, y_test = split_temporal(X, y)

    model = generate_lstm_model(
        units_1, dropout, units_2, learning_rate, X_train
    )

    model.fit(
        X_train,
        y_train,
        validation_data=(X_valid, y_valid),
        epochs=100,
        batch_size=batch_size,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=10,
                restore_best_weights=True
            )
        ],
        verbose=0
    )

    pred_scaled = model.predict(X_test, verbose=0)

    naive_scaled = X_test[:, -1, 0]

    dummy_pred = np.zeros((len(pred_scaled), len(features)))
    dummy_real = np.zeros((len(y_test), len(features)))
    dummy_naive = np.zeros((len(naive_scaled), len(features)))

    dummy_pred[:, 0] = pred_scaled.ravel()
    dummy_real[:, 0] = y_test
    dummy_naive[:, 0] = naive_scaled

    pred = scaler.inverse_transform(dummy_pred)[:, 0]
    real = scaler.inverse_transform(dummy_real)[:, 0]
    naive = scaler.inverse_transform(dummy_naive)[:, 0]

    mae = mean_absolute_error(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    mape = mean_absolute_percentage_error(real, pred) * 100
    r2 = r2_score(real, pred)
    sm = smape(real, pred)

    mae_n = mean_absolute_error(real, naive)
    rmse_n = np.sqrt(mean_squared_error(real, naive))
    mape_n = mean_absolute_percentage_error(real, naive) * 100

    skill = 1 - rmse / rmse_n

    resultados.append({
        "Ticker": ticker,
        "Media": media_close,
        "Mediana": mediana_close,
        "Desvio": desvio_close,
        "CV (%)": cv,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape,
        "sMAPE (%)": sm,
        "R2": r2,
        "MAE Naive": mae_n,
        "RMSE Naive": rmse_n,
        "MAPE Naive (%)": mape_n,
        "Skill RMSE": skill,
        "MAE/Media (%)": mae / media_close * 100,
        "RMSE/Media (%)": rmse / media_close * 100,
        "Melhor que Naive": skill > 0
    })

resultado_final = pd.DataFrame(resultados).sort_values(
    "Skill RMSE", ascending=False
)

display(resultado_final)
resultado_final.to_excel(
    "dados/refined/resultado_modelos_lstm.xlsx",
    index=False
)

  0%|          | 0/6 [00:00<?, ?it/s]g:\Meu Drive\5. Cursos\Pós ML Engineering\Fase 4 - Deep Learning e IA\lstm-stock-predictor-api\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Processando para RENT3
Treino     : (1602, 180, 21)
Validação  : (343, 180, 21)
Teste      : (344, 180, 21)


 17%|█▋        | 1/6 [01:23<06:57, 83.48s/it]g:\Meu Drive\5. Cursos\Pós ML Engineering\Fase 4 - Deep Learning e IA\lstm-stock-predictor-api\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Processando para LREN3
Treino     : (1602, 180, 21)
Validação  : (343, 180, 21)
Teste      : (344, 180, 21)


 33%|███▎      | 2/6 [02:14<04:18, 64.67s/it]g:\Meu Drive\5. Cursos\Pós ML Engineering\Fase 4 - Deep Learning e IA\lstm-stock-predictor-api\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

Processando para VBBR3
Treino     : (1302, 180, 21)
Validação  : (279, 180, 21)
Teste      : (279, 180, 21)


 83%|████████▎ | 5/6 [05:35<01:10, 70.70s/it]g:\Meu Drive\5. Cursos\Pós ML Engineering\Fase 4 - Deep Learning e IA\lstm-stock-predictor-api\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Processando para ABEV3
Treino     : (1644, 120, 21)
Validação  : (352, 120, 21)
Teste      : (353, 120, 21)


100%|██████████| 6/6 [06:43<00:00, 67.32s/it]


,Ticker,Media,Mediana,Desvio,CV (%),MAE,RMSE,MAPE (%),sMAPE (%),R2,MAE Naive,RMSE Naive,MAPE Naive (%),Skill RMSE,MAE/Media (%),RMSE/Media (%),Melhor que Naive
0,RENT3,36.02,38.82,14.95,41.49,5.57,6.69,14.46,14.54,-0.08,0.73,0.99,1.88,-5.76,15.46,18.56,False
1,LREN3,19.48,17.65,6.83,35.04,2.13,2.54,14.10,15.34,-0.94,0.27,0.36,1.95,-6.12,10.91,13.05,False
2,SMFT3,19.01,19.63,3.97,20.86,3.97,4.68,17.26,19.48,-2.43,0.38,0.51,1.79,-8.18,20.91,24.60,False
5,ABEV3,12.43,12.33,1.55,12.48,2.03,2.50,14.45,15.99,-1.18,0.14,0.21,1.05,-10.64,16.33,20.09,False
3,MULT3,20.60,19.88,4.10,19.88,7.37,8.37,25.89,30.70,-3.43,0.33,0.45,1.23,-17.68,35.75,40.61,False
4,VBBR3,16.12,15.20,4.44,27.57,8.80,10.32,33.97,42.75,-2.68,0.32,0.45,1.34,-22.16,54.62,64.01,False
